# Notebook 6: CNNs for Medical Imaging — Finding Pneumonia in Chest X-rays

**Goes with:** the slide deck *CNNs for Medical Imaging: Teaching a Computer to Read Chest X-rays*.

In this notebook, we build the complete project from the slides, step by step:

| Notebook part | What we do | Slides |
|---|---|---|
| 1. Data | Download and explore a real, open dataset of children's chest X-rays | Part 3 |
| 2. Prepare | Scale the pixels and set up data augmentation | Part 4 |
| 3. Build | Design a convolutional neural network (CNN) | Part 4 |
| 4. Train | Train it with class weights and early stopping | Part 4 |
| 5. Evaluate | Confusion matrix, sensitivity, specificity, thresholds, ROC and AUC | Part 5 |
| 6. Explain | Learned filters, feature maps, Grad-CAM heat maps, and mistakes | Part 6 |

> ⚠️ **Important:** This is a **learning project**, not a medical tool. It must never be used to make decisions about anyone's health. Real medical AI needs years of testing, doctors in charge, and approval from health regulators (slides, Part 7).

**How to run:** This notebook works best in **Google Colab** (free). Choose *Runtime → Change runtime type → GPU* to make training faster (optional). Then run each cell from top to bottom with **Shift + Enter**. Training takes about 2 to 5 minutes.

## 0. Setup

We load the libraries (ready-made tools) we need. All of them are free and open source.

In [ ]:
import os
# Hide most of TensorFlow's long technical start-up messages.
# You may still see a few lines starting with "I0000" or "WARNING".
# They are information messages, NOT errors. You can safely ignore them.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np                    # numbers and arrays (grids of numbers)
import pandas as pd                   # neat tables
import matplotlib.pyplot as plt       # charts and pictures
import tensorflow as tf               # the deep learning engine
from tensorflow import keras          # the easy-to-use part of TensorFlow
from tensorflow.keras import layers   # building blocks: Conv2D, Dense, ...

# Tools from scikit-learn for measuring how good our model is.
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

# Fix ALL the random numbers, so we get (almost) the same result every run.
# (Tiny differences can still happen between different computers.)
keras.utils.set_random_seed(42)

# Our color scheme, matching the slides.
TEAL, ORANGE, CORAL, PURPLE, NAVY = "#0FA3A3", "#F28C28", "#E5566B", "#6C5CE7", "#16213E"
plt.rcParams["font.size"] = 12

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)

---
# Part 1 · The Data (slides, Part 3)

## 1.1 Download the dataset

We use **PneumoniaMNIST**, part of the open **MedMNIST v2** collection.

* **What it is:** 5,856 chest X-rays of young children, labeled **normal** or **pneumonia**.
* **Image size:** 64 × 64 pixels, grayscale.
* **Where it comes from:** The original X-rays were published by Kermany and colleagues (2018, *Cell*). The MedMNIST team (Yang and colleagues, 2023, *Scientific Data*) resized them and split them into training, validation, and test sets.
* **License:** CC BY 4.0 (free to use, as long as we give credit).
* **Download:** about 20 MB, from Zenodo (an open science archive), the first time only.

The cell below lets you switch to another dataset later (see the challenge at the end). For now, keep `DATASET = "pneumonia"`.

In [ ]:
# ------------------------------------------------------------------
# Open-access datasets from MedMNIST v2 (all CC BY 4.0).
# Each entry tells the notebook where to download the file, the
# "fingerprint" (MD5) that proves the file is not damaged, and the
# names of the two classes.
# ------------------------------------------------------------------
DATASETS = {
    "pneumonia": {
        "file": "pneumoniamnist_64.npz",
        "url": "https://zenodo.org/records/10519652/files/pneumoniamnist_64.npz?download=1",
        "md5": "8f4eceb4ccffa70c672198ea285246c6",
        "names": ["NORMAL", "PNEUMONIA"],   # label 0, label 1
        "flip": False,                       # labels are already 0 = healthy, 1 = disease
    },
    "breast": {
        "file": "breastmnist_64.npz",
        "url": "https://zenodo.org/records/10519652/files/breastmnist_64.npz?download=1",
        "md5": "742edef2a1fd1524b2efff4bd7ba9364",
        "names": ["NORMAL or BENIGN", "MALIGNANT"],
        # In the original BreastMNIST, 0 = malignant and 1 = normal/benign.
        # We flip them so that 1 always means "the disease we want to find".
        "flip": True,
    },
}

# ===== CHANGE THIS LINE TO TRY ANOTHER DATASET =====
DATASET = "pneumonia"
# ===================================================

info = DATASETS[DATASET]
CLASS_NAMES = info["names"]

def download_dataset(info):
    """
    Download the .npz file (a compressed bundle of NumPy arrays) and
    return its local path.

    keras.utils.get_file() downloads the file ONCE and keeps a copy in a
    cache folder, so running this cell again is instant. It also checks
    the MD5 fingerprint, so a broken download is caught right away.
    """
    # If you uploaded the file yourself (for example, with no internet
    # in class), use that copy.
    if os.path.exists(info["file"]):
        print("Using the file found in this folder:", info["file"])
        return info["file"]
    try:
        return keras.utils.get_file(fname=info["file"], origin=info["url"], file_hash=info["md5"])
    except Exception as error:
        raise RuntimeError(
            "Could not download the dataset. Check your internet connection.\n"
            f"You can also download it yourself from:\n  {info['url']}\n"
            "then upload the file into the same folder as this notebook "
            "(in Colab: the folder icon on the left -> Upload), and run this cell again."
        ) from error

path = download_dataset(info)
data = np.load(path)          # opens the bundle; it works like a dictionary of arrays
print("Downloaded to:", path)
print("Arrays inside the file:", data.files)

## 1.2 Unpack the three piles of X-rays (slide 16)

The file already contains three separate piles:

* **Training set:** practice problems. The network learns from these.
* **Validation set:** practice quizzes. We use them to choose the best epoch and the threshold.
* **Test set:** the final exam. We use it **only once**, at the very end.

In [ ]:
def get_split(name):
    """Return the images and labels of one split ('train', 'val' or 'test')."""
    images = data[f"{name}_images"]            # shape: (number of images, 64, 64)
    labels = data[f"{name}_labels"].ravel()    # shape: (number of images,)  ravel() makes it a flat list
    if info["flip"]:
        labels = 1 - labels                    # swap 0 <-> 1 (only for datasets that need it)
    return images, labels

x_train_raw, y_train = get_split("train")
x_val_raw, y_val = get_split("val")
x_test_raw, y_test = get_split("test")

# Count how many images of each class are in each pile.
summary = pd.DataFrame({
    "split": ["training", "validation", "test"],
    "images": [len(y_train), len(y_val), len(y_test)],
    CLASS_NAMES[0]: [np.sum(y == 0) for y in (y_train, y_val, y_test)],
    CLASS_NAMES[1]: [np.sum(y == 1) for y in (y_train, y_val, y_test)],
})
print("Image shape:", x_train_raw.shape[1:], " pixel type:", x_train_raw.dtype)
summary

## 1.3 Look at the X-rays (slide 14)

Always look at your data before training! Can **you** spot the difference between the two rows?

In [ ]:
rng = np.random.default_rng(3)     # a random number generator with a fixed seed

fig, axes = plt.subplots(2, 5, figsize=(13, 5.8))
for row, (label, color) in enumerate([(0, TEAL), (1, CORAL)]):
    # Pick 5 random training images that have this label.
    picks = rng.choice(np.where(y_train == label)[0], size=5, replace=False)
    for col, index in enumerate(picks):
        ax = axes[row, col]
        ax.imshow(x_train_raw[index], cmap="gray")   # draw in grayscale
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():             # colored frame around each picture
            spine.set_edgecolor(color); spine.set_linewidth(4)
        if col == 0:
            ax.set_ylabel(CLASS_NAMES[label], fontsize=14, fontweight="bold", color=color)
plt.suptitle("Training X-rays: top row normal, bottom row pneumonia", fontweight="bold")
plt.tight_layout()
plt.show()

## 1.4 An X-ray is just numbers (slide 15)

Each pixel is one number from **0 (black)** to **255 (white)**. Let us zoom into an 8 × 8 patch.

In [ ]:
image = x_train_raw[np.where(y_train == 0)[0][5]]     # one normal X-ray

# Find the 8 x 8 patch with the most contrast (where it crosses an edge).
best_contrast, top, left = max(
    (image[r:r + 8, c:c + 8].std(), r, c) for r in range(12, 44) for c in range(8, 48))
patch = image[top:top + 8, left:left + 8]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(image, cmap="gray")
axes[0].add_patch(plt.Rectangle((left - 0.5, top - 0.5), 8, 8, fill=False, edgecolor=ORANGE, linewidth=3))
axes[0].set_title("The whole X-ray (orange box = our patch)")
axes[1].imshow(patch, cmap="gray", vmin=0, vmax=255)
for (r, c), value in np.ndenumerate(patch):            # write each number on its pixel
    axes[1].text(c, r, value, ha="center", va="center", fontsize=10,
                 color="black" if value > 128 else "white")
axes[1].set_title("Zoomed in: 64 numbers")
for ax in axes: ax.axis("off")
plt.show()

print(f"One image has {image.shape[0]} x {image.shape[1]} = {image.size:,} numbers.")

## 1.5 The hidden trap: unbalanced data (slide 17)

About 3 out of 4 training X-rays show pneumonia. A "lazy" model that **always** answers "pneumonia" would still get a decent accuracy, without looking at a single X-ray. Let us calculate it.

In [ ]:
share_pneumonia = np.mean(y_train == 1) * 100
print(f"Training X-rays with {CLASS_NAMES[1]}: {share_pneumonia:.1f}%")

# The lazy model answers "1" for every test X-ray.
lazy_predictions = np.ones_like(y_test)
lazy_accuracy = np.mean(lazy_predictions == y_test) * 100
print(f"Lazy model accuracy on the test set: {lazy_accuracy:.1f}%")
print(f"...but it gets every single {CLASS_NAMES[0]} X-ray wrong ({np.sum(y_test == 0)} of them)!")
print()
print("Remember this number. Our CNN must do MUCH better than this to be useful.")

---
# Part 2 · Prepare the Images (slides 19 and 20)

## 2.1 Scale the pixels and add a channel

* Divide every pixel by 255, so values go from **0 to 1**. Networks learn best with small numbers.
* Add a **channel** dimension. Color photos have 3 channels (red, green, blue); X-rays have 1. So each image becomes 64 × 64 × **1**.

In [ ]:
def prepare(images):
    """Scale pixels to 0..1 and add a channel dimension."""
    images = images.astype("float32") / 255.0      # whole numbers -> decimals between 0 and 1
    if images.ndim == 3:                           # grayscale: (n, 64, 64) -> (n, 64, 64, 1)
        images = images[..., np.newaxis]
    return images

x_train = prepare(x_train_raw)
x_val = prepare(x_val_raw)
x_test = prepare(x_test_raw)

print("Before:", x_train_raw.shape, "values from", x_train_raw.min(), "to", x_train_raw.max())
print("After: ", x_train.shape, "values from", x_train.min(), "to", x_train.max())

## 2.2 Data augmentation: more variety for free (slide 20)

Each time the network sees a training X-ray, we change it a little:

* **Rotate** by up to about 11° (`0.03` of a full turn)
* **Shift** up and down or left and right by up to 5%
* **Zoom** in or out by up to 10%

We **never flip** chest X-rays left to right: the heart is on the left side, so a flipped chest would not look real.

These layers are only active **during training**. During testing, they do nothing.

In [ ]:
augmentation = keras.Sequential([
    layers.RandomRotation(0.03, name="aug_rotate"),          # +/- 0.03 x 360° ≈ 11°
    layers.RandomTranslation(0.05, 0.05, name="aug_shift"),  # +/- 5% up/down, left/right
    layers.RandomZoom(0.1, name="aug_zoom"),                 # +/- 10% zoom
], name="augmentation")

# Show one X-ray and 5 random augmented versions of it.
example = x_train[np.where(y_train == 0)[0][11]][np.newaxis]   # shape (1, 64, 64, 1)
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(example[0, :, :, 0], cmap="gray"); axes[0].set_title("Original", fontweight="bold")
for k in range(1, 6):
    # training=True switches augmentation ON, so we can see it.
    augmented = augmentation(example, training=True).numpy()
    axes[k].imshow(augmented[0, :, :, 0], cmap="gray"); axes[k].set_title(f"Version {k}", color=PURPLE)
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

---
# Part 3 · Build the CNN (slide 21)

Our network follows the classic pattern: **Conv → Pool → Conv → Pool → Conv → Pool**, then **Flatten → Dropout → Dense → Output**.

| Layer | What it does | Output shape |
|---|---|---|
| Input | One grayscale X-ray | 64 × 64 × 1 |
| Conv2D (16 filters) + ReLU | Finds 16 simple patterns, like edges | 64 × 64 × 16 |
| MaxPooling | Halves the size, keeps the strongest signals | 32 × 32 × 16 |
| Conv2D (32 filters) + ReLU | Combines edges into shapes | 32 × 32 × 32 |
| MaxPooling | Halves the size again | 16 × 16 × 32 |
| Conv2D (64 filters) + ReLU | Finds larger patterns, like hazy areas | 16 × 16 × 64 |
| MaxPooling | Halves the size again | 8 × 8 × 64 |
| Flatten | Unrolls the grid into one list | 4,096 |
| Dropout (30%) | Randomly switches off 30% of the values during training | 4,096 |
| Dense (32) + ReLU | Combines all the features | 32 |
| Dense (1) + sigmoid | The **chance of disease**, from 0 to 1 | 1 |

In [ ]:
def build_cnn(input_shape):
    """Build and return our CNN. input_shape is (height, width, channels)."""
    inputs = keras.Input(shape=input_shape, name="xray")

    # Data augmentation (only active during training).
    x = augmentation(inputs)

    # ---- Block 1: 16 filters of size 3 x 3 ----
    # padding="same" adds a border of zeros so the image keeps its size.
    x = layers.Conv2D(16, kernel_size=3, padding="same", activation="relu", name="conv1")(x)
    x = layers.MaxPooling2D(pool_size=2, name="pool1")(x)        # 64x64 -> 32x32

    # ---- Block 2: 32 filters ----
    x = layers.Conv2D(32, kernel_size=3, padding="same", activation="relu", name="conv2")(x)
    x = layers.MaxPooling2D(pool_size=2, name="pool2")(x)        # 32x32 -> 16x16

    # ---- Block 3: 64 filters ----
    # We will look inside this layer later with Grad-CAM.
    x = layers.Conv2D(64, kernel_size=3, padding="same", activation="relu", name="conv3")(x)
    x = layers.MaxPooling2D(pool_size=2, name="pool3")(x)        # 16x16 -> 8x8

    # ---- Decide ----
    x = layers.Flatten(name="flatten")(x)                        # 8 x 8 x 64 = 4,096 numbers
    x = layers.Dropout(0.3, name="dropout")(x)                   # fight memorizing (slide 22)
    x = layers.Dense(32, activation="relu", name="dense")(x)
    # One output neuron with sigmoid = a chance between 0 and 1.
    outputs = layers.Dense(1, activation="sigmoid", name="disease_chance")(x)

    return keras.Model(inputs, outputs, name="medical_cnn")

model = build_cnn(x_train.shape[1:])
model.summary()

**Where do the parameter counts come from?** Take `conv1`: each of its 16 filters has 3 × 3 = 9 weights, plus 1 bias. That makes 16 × (9 + 1) = **160** knobs. The `dense` layer is the biggest: 32 neurons × (4,096 inputs + 1 bias) = **131,104** knobs. Training will tune all of them automatically.

---
# Part 4 · Train the CNN (slides 22 and 23)

## 4.1 Class weights (slide 17)

To fix the unbalanced data, mistakes on the smaller class must count more. The usual formula is:

$$\text{weight of a class} = \frac{\text{total number of images}}{2 \times \text{number of images in that class}}$$

In [ ]:
n_total = len(y_train)
n_class0 = np.sum(y_train == 0)
n_class1 = np.sum(y_train == 1)

class_weight = {
    0: n_total / (2 * n_class0),   # the smaller class gets a weight above 1
    1: n_total / (2 * n_class1),   # the bigger class gets a weight below 1
}
print(f"Weight for {CLASS_NAMES[0]}: {class_weight[0]:.2f}")
print(f"Weight for {CLASS_NAMES[1]}: {class_weight[1]:.2f}")
print("Check: weight x count is the same for both classes:",
      round(class_weight[0] * n_class0), "and", round(class_weight[1] * n_class1))

## 4.2 Compile: choose how the network learns

* **Optimizer `Adam`:** a smart version of gradient descent that adjusts its own step size.
* **Loss `binary_crossentropy`:** the standard loss for yes/no questions. Lower is better.
* **Metrics:** accuracy, **AUC** (slide 31), and **sensitivity** (called *recall* in Keras: of all the sick cases, how many did we catch?).

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy",
             keras.metrics.AUC(name="auc"),
             keras.metrics.Recall(name="sensitivity")],
)
print("Model compiled and ready to train.")

## 4.3 Train with early stopping (slides 22 and 23)

* **`epochs=25`:** at most 25 passes through the training X-rays.
* **`batch_size=64`:** update the weights after every group of 64 X-rays.
* **Early stopping:** after each epoch, we check the **validation AUC**. If it does not improve for 6 epochs in a row, training stops, and we go back to the **best** epoch (`restore_best_weights=True`).

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_auc",           # watch the AUC on the validation set
    mode="max",                  # higher AUC is better
    patience=6,                  # wait 6 epochs without improvement before stopping
    restore_best_weights=True,   # go back to the best epoch at the end
)

history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),   # the "practice quiz" after every epoch
    epochs=25,
    batch_size=64,
    class_weight=class_weight,        # our fix for unbalanced data
    callbacks=[early_stopping],
    verbose=2,                        # print one line per epoch
)

best_epoch = int(np.argmax(history.history["val_auc"])) + 1
print(f"\nTraining ran for {len(history.history['loss'])} epochs. Best epoch: {best_epoch} (we kept this one).")

## 4.4 Watch it train (slide 23)

In [ ]:
epochs = np.arange(1, len(history.history["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
for ax, key, title in [(axes[0], "loss", "Loss goes down"), (axes[1], "auc", "AUC goes up")]:
    ax.plot(epochs, history.history[key], "o-", color=TEAL, label="training")
    ax.plot(epochs, history.history["val_" + key], "o--", color=ORANGE, label="validation")
    ax.axvline(best_epoch, color="gray", linestyle=":", label=f"best epoch ({best_epoch})")
    ax.set_title(title, fontweight="bold"); ax.set_xlabel("epoch"); ax.set_ylabel(key)
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()
plt.show()

**Two things to notice (slide 23):**

* The validation curve is **bumpy**. The validation set is small, so a few changed answers move the score a lot.
* Validation can look **better** than training. That is because dropout and augmentation make training harder on purpose, and they are switched off for validation.

---
# Part 5 · Evaluate Like a Doctor (slides 25 to 32)

## 5.1 Ask the network about the test X-rays

For each X-ray, the network outputs a **chance** from 0 to 1. We decide with a **threshold**: above it, we say "disease"; below it, "normal". We start with the usual threshold, **0.5**.

In [ ]:
# predict() returns one chance per X-ray. ravel() turns the column into a flat list.
test_chance = model.predict(x_test, verbose=0).ravel()
val_chance = model.predict(x_val, verbose=0).ravel()

def evaluate(y_true, chance, threshold):
    """Turn chances into answers with a threshold, then count every kind of result."""
    predicted = (chance >= threshold).astype(int)
    # confusion_matrix gives [[TN, FP], [FN, TP]]; ravel() lists them in that order.
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "threshold": threshold,
        "missed (FN)": fn,                          # sick, but we said normal
        "false alarms (FP)": fp,                    # healthy, but we said sick
        "sensitivity %": round(tp / (tp + fn) * 100, 1),  # sick people caught
        "specificity %": round(tn / (tn + fp) * 100, 1),  # healthy people cleared
        "accuracy %": round((tp + tn) / len(y_true) * 100, 1),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    }

result_05 = evaluate(y_test, test_chance, 0.5)
print(f"Test accuracy at threshold 0.5: {result_05['accuracy %']}%   (lazy model: {lazy_accuracy:.1f}%)")

## 5.2 The confusion matrix (slide 26)

Rows show the **truth**. Columns show what the **network said**.

In [ ]:
def draw_confusion(result, title):
    """Draw a colored 2 x 2 confusion matrix with friendly labels."""
    cells = [[result["TN"], result["FP"]], [result["FN"], result["TP"]]]
    labels = [["correct", "false alarm"], ["MISSED", "caught"]]
    colors = [[TEAL, ORANGE], [CORAL, TEAL]]
    fig, ax = plt.subplots(figsize=(5.8, 5))
    for r in range(2):
        for c in range(2):
            ax.add_patch(plt.Rectangle((c, 1 - r), 1, 1, color=colors[r][c], alpha=0.18))
            ax.text(c + 0.5, 1.55 - r, cells[r][c], ha="center", va="center",
                    fontsize=30, fontweight="bold", color=colors[r][c])
            ax.text(c + 0.5, 1.25 - r, labels[r][c], ha="center", va="center",
                    fontsize=12, fontweight="bold", color=colors[r][c])
    ax.set_xlim(0, 2); ax.set_ylim(0, 2)
    ax.set_xticks([0.5, 1.5]); ax.set_xticklabels([f"said\n{n}" for n in CLASS_NAMES])
    ax.set_yticks([1.5, 0.5]); ax.set_yticklabels([f"really\n{n}" for n in CLASS_NAMES])
    ax.set_title(title, fontweight="bold")
    plt.show()

draw_confusion(result_05, f"Test set, threshold 0.5 ({len(y_test)} X-rays)")

## 5.3 Sensitivity and specificity (slide 27)

* **Sensitivity** = caught ÷ all sick = TP ÷ (TP + FN). *How many sick people did we find?*
* **Specificity** = cleared ÷ all healthy = TN ÷ (TN + FP). *How many healthy people did we clear?*

For a **screening tool**, sensitivity matters most: missing a sick child is worse than a false alarm.

In [ ]:
r = result_05
print(f"Sensitivity = {r['TP']} / ({r['TP']} + {r['FN']}) = {r['sensitivity %']}%")
print(f"Specificity = {r['TN']} / ({r['TN']} + {r['FP']}) = {r['specificity %']}%")

## 5.4 The network gives a chance, not an answer (slide 28)

This chart shows the chance the network gave to every test X-ray. Most are easy: very close to 0 or to 1. The hard ones sit in the middle, close to the threshold.

In [ ]:
bins = np.linspace(0, 1, 26)
plt.figure(figsize=(10, 4.2))
plt.hist(test_chance[y_test == 0], bins=bins, color=TEAL, alpha=0.85, label=f"really {CLASS_NAMES[0]}")
plt.hist(test_chance[y_test == 1], bins=bins, color=CORAL, alpha=0.7, label=f"really {CLASS_NAMES[1]}")
plt.axvline(0.5, color=NAVY, linewidth=2.5, label="threshold 0.5")
plt.xlabel("the network's chance of disease"); plt.ylabel("number of test X-rays")
plt.legend(); plt.grid(alpha=0.2)
plt.show()

## 5.5 Moving the threshold (slide 29)

What happens if we move the threshold? Lower thresholds catch more sick children but cause more false alarms.

In [ ]:
columns = ["threshold", "missed (FN)", "false alarms (FP)", "sensitivity %", "specificity %", "accuracy %"]
table = pd.DataFrame([evaluate(y_test, test_chance, t) for t in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]])[columns]
table

## 5.6 Choosing a threshold, the careful way (slide 30)

**Our rule:** catch at least **95%** of sick cases.

We must choose the threshold using the **validation set**, not the test set. If we chose it by looking at the test set, our final score would no longer be honest.

In [ ]:
TARGET_SENSITIVITY = 0.95   # catch at least 95% of sick cases

# Try every possible threshold, from the highest chance down to the lowest.
# The first (highest) threshold that reaches the target is our choice:
# it reaches the goal with as few false alarms as possible.
chosen_threshold = None
for t in np.sort(np.unique(val_chance))[::-1]:
    sens = np.mean(val_chance[y_val == 1] >= t)     # share of sick validation cases caught
    if sens >= TARGET_SENSITIVITY:
        chosen_threshold = float(t)
        break

print(f"Threshold chosen on the VALIDATION set: {chosen_threshold:.2f}")

# Only now do we check the result on the test set.
result_low = evaluate(y_test, test_chance, chosen_threshold)
comparison = pd.DataFrame([result_05, result_low])[columns]
comparison["threshold"] = comparison["threshold"].round(2)
comparison

In [ ]:
# Draw sensitivity and specificity for every threshold (on the test set).
thresholds = np.linspace(0.01, 0.99, 99)
sens_curve = [evaluate(y_test, test_chance, t)["sensitivity %"] for t in thresholds]
spec_curve = [evaluate(y_test, test_chance, t)["specificity %"] for t in thresholds]

plt.figure(figsize=(9, 4.3))
plt.plot(thresholds, sens_curve, color=CORAL, linewidth=3, label="sensitivity (sick cases caught)")
plt.plot(thresholds, spec_curve, color=TEAL, linewidth=3, label="specificity (healthy cases cleared)")
plt.axvline(0.5, color=NAVY, linestyle="--", label="default 0.5")
plt.axvline(chosen_threshold, color=ORANGE, linestyle="--", label=f"our choice {chosen_threshold:.2f}")
plt.xlabel("threshold"); plt.ylabel("percent"); plt.ylim(40, 102)
plt.legend(loc="lower center"); plt.grid(alpha=0.3)
plt.show()

print(f"We traded {result_low['FP'] - result_05['FP']} extra false alarms "
      f"for {result_05['FN'] - result_low['FN']} fewer missed sick cases.")

## 5.7 The ROC curve and AUC (slide 31)

The ROC curve shows **every** threshold at once. AUC is the **area under the curve**: 0.5 is a coin flip, and 1.0 is perfect.

In [ ]:
false_alarm_rate, catch_rate, _ = roc_curve(y_test, test_chance)
test_auc = roc_auc_score(y_test, test_chance)
val_auc = roc_auc_score(y_val, val_chance)

plt.figure(figsize=(5.8, 5.4))
plt.fill_between(false_alarm_rate, catch_rate, color=TEAL, alpha=0.15)
plt.plot(false_alarm_rate, catch_rate, color=TEAL, linewidth=3, label=f"our CNN (AUC = {test_auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="coin flip (AUC = 0.5)")
for res, color in [(result_05, NAVY), (result_low, ORANGE)]:
    plt.scatter(1 - res["specificity %"] / 100, res["sensitivity %"] / 100, s=120,
                color=color, edgecolor="white", zorder=5, label=f"threshold {res['threshold']:.2f}")
plt.xlabel("false alarm rate (1 - specificity)"); plt.ylabel("sensitivity")
plt.legend(loc="lower right"); plt.grid(alpha=0.3)
plt.show()

print(f"Validation AUC: {val_auc:.3f}")
print(f"Test AUC:       {test_auc:.3f}")

**A warning sign (slide 32):** the test AUC is usually **lower** than the validation AUC. In PneumoniaMNIST, the test X-rays come from a different part of the original collection than the training X-rays. New data is often a little different, and real hospitals are even more different. Medical AI must be tested on data from many places.

---
# Part 6 · Opening the Black Box (slides 34 to 37)

## 6.1 What did the first layer learn? (slide 34)

These 16 filters started as random numbers. Training shaped them into pattern detectors.

In [ ]:
# get_weights() returns [filters, biases]. The filters have shape (3, 3, channels, 16).
filters, biases = model.get_layer("conv1").get_weights()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for k, ax in enumerate(axes.flat):
    ax.imshow(filters[:, :, 0, k], cmap="coolwarm")   # red = positive, blue = negative
    ax.set_title(f"filter {k + 1}", fontsize=9); ax.axis("off")
plt.suptitle("The 16 filters learned by conv1 (red = +, blue = -)", fontweight="bold")
plt.tight_layout()
plt.show()

### Feature maps: one X-ray through the learned filters

In [ ]:
# A small "spy" model that stops after conv1, so we can see its output.
conv1_model = keras.Model(inputs=model.inputs, outputs=model.get_layer("conv1").output)

example_index = np.where(y_test == 1)[0][4]
feature_maps = conv1_model([x_test[example_index:example_index + 1]], training=False).numpy()[0]

fig, axes = plt.subplots(2, 9, figsize=(16, 4.4))
axes.flat[0].imshow(x_test[example_index, :, :, 0], cmap="gray")
axes.flat[0].set_title("input", fontweight="bold")
for k in range(16):
    axes.flat[k + 1].imshow(feature_maps[:, :, k], cmap="magma")
    axes.flat[k + 1].set_title(f"map {k + 1}", fontsize=9)
for ax in axes.flat: ax.axis("off")
plt.suptitle("Bright areas = where each filter found its pattern", fontweight="bold")
plt.tight_layout()
plt.show()

**Why are some maps completely dark?** Those filters did not find their pattern anywhere in this X-ray. ReLU turned all their negative values into 0, so the whole map is black. Other X-rays may light them up.

## 6.2 Grad-CAM: where is the network looking? (slide 35)

**Grad-CAM** builds a heat map in four steps:

1. Run the X-ray through the network and keep the feature maps of the last conv layer (`conv3`).
2. Ask: *"If this feature map got brighter, how much would the disease score go up?"* That is the **gradient**, the same idea as in backpropagation.
3. Give each feature map a weight equal to its average gradient, then add the weighted maps together.
4. Keep only the positive parts (ReLU) and stretch the result to the size of the X-ray.

Warm colors show the areas that pushed the network **toward disease**.

In [ ]:
# A model with TWO outputs: the conv3 feature maps AND the final chance.
grad_model = keras.Model(inputs=model.inputs,
                         outputs=[model.get_layer("conv3").output, model.output])

def grad_cam(image):
    """Return a 64 x 64 heat map (values 0 to 1) for one prepared X-ray."""
    x = tf.convert_to_tensor(image[np.newaxis])            # make a batch of one image
    with tf.GradientTape() as tape:                        # records the math so we can get gradients
        conv_maps, prediction = grad_model([x], training=False)
        chance = tf.clip_by_value(prediction[:, 0], 1e-7, 1 - 1e-7)
        # We use the score BEFORE the sigmoid squashed it (called the "logit").
        # When the network is ~100% sure, the sigmoid curve is flat, so its
        # gradient is almost 0 and the heat map would fade. The logit avoids that.
        score = tf.math.log(chance / (1 - chance))
    # Step 2: how much does the score change when each pixel of each map changes?
    gradients = tape.gradient(score, conv_maps)[0]         # shape (16, 16, 64)
    # Step 3: one importance weight per feature map = its average gradient.
    weights = tf.reduce_mean(gradients, axis=(0, 1))       # shape (64,)
    heatmap = tf.reduce_sum(conv_maps[0] * weights, axis=-1)
    # Step 4: keep positives only, scale to 0..1, resize to the X-ray size.
    heatmap = tf.nn.relu(heatmap).numpy()
    heatmap = heatmap / (heatmap.max() + 1e-8)             # + tiny number avoids dividing by 0
    return tf.image.resize(heatmap[..., np.newaxis], image.shape[:2]).numpy()[..., 0]

predicted_05 = (test_chance >= 0.5).astype(int)

def pick(mask, position):
    """Return the index of the n-th X-ray that matches a condition (or the last one, if fewer exist)."""
    matches = np.where(mask)[0]
    return matches[min(position, len(matches) - 1)] if len(matches) > 0 else None

examples = [
    ("caught", pick((y_test == 1) & (predicted_05 == 1), 2)),
    ("caught", pick((y_test == 1) & (predicted_05 == 1), 9)),
    ("cleared", pick((y_test == 0) & (predicted_05 == 0), 3)),
    ("MISSED", pick((y_test == 1) & (predicted_05 == 0), 1)),
]
examples = [(kind, i) for kind, i in examples if i is not None]   # skip any kind that does not exist

fig, axes = plt.subplots(2, len(examples), figsize=(3.3 * len(examples), 6.6), squeeze=False)
for col, (kind, i) in enumerate(examples):
    axes[0, col].imshow(x_test[i, :, :, 0], cmap="gray")
    axes[0, col].set_title(f"really {CLASS_NAMES[y_test[i]]}, {kind}\nchance: {test_chance[i]*100:.0f}%",
                           color=CORAL if kind == "MISSED" else NAVY, fontweight="bold", fontsize=11)
    axes[1, col].imshow(x_test[i, :, :, 0], cmap="gray")
    axes[1, col].imshow(grad_cam(x_test[i]), cmap="jet", alpha=0.45)   # heat map on top
for ax in axes.flat: ax.axis("off")
plt.tight_layout()
plt.show()

**Be careful with heat maps (slide 37).** If the warm area sits on a text marker, a wire, or the edge of the picture instead of the lungs, the network may have learned a **shortcut**. A high score is not proof: we must check *why* the model is right.

## 6.3 Learning from mistakes (slide 36)

In [ ]:
missed = np.where((y_test == 1) & (predicted_05 == 0))[0][:4]        # sick, but called normal
false_alarms = np.where((y_test == 0) & (predicted_05 == 1))[0][:4]  # healthy, but flagged

print(f"Missed: {np.sum((y_test == 1) & (predicted_05 == 0))}   False alarms: {np.sum((y_test == 0) & (predicted_05 == 1))}")
fig, axes = plt.subplots(2, 4, figsize=(12, 6.4))
for ax in axes.flat: ax.axis("off")            # start with empty boxes
for row, (indices, kind, color) in enumerate([(missed, "missed", CORAL), (false_alarms, "false alarm", ORANGE)]):
    for col, i in enumerate(indices):
        axes[row, col].imshow(x_test[i, :, :, 0], cmap="gray"); axes[row, col].axis("off")
        axes[row, col].set_title(f"{kind}\nchance {test_chance[i]*100:.0f}%", color=color, fontweight="bold")
plt.tight_layout()
plt.show()

## 6.4 Ask the network about one X-ray

This function shows how a doctor-facing tool *might* present a result: the chance, the answer at our chosen threshold, and a heat map. Remember: this is a **learning project**, not a diagnosis.

In [ ]:
def explain_xray(index, threshold=chosen_threshold):
    """Show one test X-ray, the network's chance, its answer, and a Grad-CAM heat map."""
    image = x_test[index]
    chance = float(model.predict(image[np.newaxis], verbose=0)[0, 0])
    answer = CLASS_NAMES[int(chance >= threshold)]
    truth = CLASS_NAMES[y_test[index]]

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1, 1, 1.2]})
    axes[0].imshow(image[:, :, 0], cmap="gray"); axes[0].set_title("X-ray"); axes[0].axis("off")
    axes[1].imshow(image[:, :, 0], cmap="gray"); axes[1].imshow(grad_cam(image), cmap="jet", alpha=0.45)
    axes[1].set_title("Grad-CAM heat map"); axes[1].axis("off")
    axes[2].barh([CLASS_NAMES[0], CLASS_NAMES[1]], [1 - chance, chance], color=[TEAL, CORAL])
    axes[2].axvline(threshold, color=ORANGE, linestyle="--", label=f"threshold {threshold:.2f}")
    axes[2].set_xlim(0, 1); axes[2].legend(loc="lower right")
    axes[2].set_title(f"Network: {answer}  |  Truth: {truth}",
                      color="green" if answer == truth else "red", fontweight="bold")
    plt.tight_layout()
    plt.show()

# ===== CHANGE THIS NUMBER TO LOOK AT A DIFFERENT TEST X-RAY (0 up to the number of test images - 1) =====
explain_xray(10)

---
# Summary

| Step | What we did | Key idea |
|---|---|---|
| Data | Loaded 5,856 open-access chest X-rays | Keep training, validation, and test sets apart |
| Prepare | Scaled pixels, added augmentation | Never flip chest X-rays |
| Build | A CNN with 3 Conv blocks | Picture shrinks, feature maps grow |
| Train | Class weights, dropout, early stopping | Fight unbalanced data and memorizing |
| Evaluate | Confusion matrix, sensitivity, specificity, ROC | The threshold is a human choice |
| Explain | Filters, feature maps, Grad-CAM, mistakes | Check *why* the model is right |

## Challenge questions

1. **Change the network:** delete the third Conv block (`conv3` and `pool3`). Remember to change Grad-CAM to use `conv2`. Does the test AUC go up or down?
2. **Turn off augmentation:** replace `x = augmentation(inputs)` with `x = inputs`. What happens to the gap between validation and test AUC?
3. **Pick your threshold:** change `TARGET_SENSITIVITY` to `0.99`. How many false alarms does it cost to catch almost every sick child?
4. **New dataset:** change `DATASET = "pneumonia"` to `DATASET = "breast"` and run everything again. This trains the same CNN on **BreastMNIST** (breast ultrasound images, CC BY 4.0). Is it easier or harder? Why might it be harder? (Hint: compare the number of training images.)
5. **Bigger challenge:** MedMNIST also has **BloodMNIST** (8 types of blood cells, color images) and **OrganAMNIST** (11 organs from CT scans). They have many classes, so the output must become `Dense(number_of_classes, activation="softmax")`, and the loss must become `sparse_categorical_crossentropy`. Find their download links and licenses at medmnist.com.

## Dataset credits

* Yang, J., Shi, R., Wei, D., Liu, Z., Zhao, L., Ke, B., Pfister, H., & Ni, B. (2023). MedMNIST v2: A large-scale lightweight benchmark for 2D and 3D biomedical image classification. *Scientific Data*, 10, 41. License: CC BY 4.0.
* Kermany, D. S., Goldbaum, M., et al. (2018). Identifying medical diagnoses and treatable diseases by image-based deep learning. *Cell*, 172(5), 1122–1131.